# test2

In [7]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
import pandas as pd
import json

THAILLM_API_KEY = "ou9erMIcBaSv0QwU9ExnIK7B1CiAJ9u0"

In [8]:
THAILLM_API_KEY = "ou9erMIcBaSv0QwU9ExnIK7B1CiAJ9u0"

In [9]:
def ask_llm(messages, model="pathumma", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,

        # ลองตัวนี้ก่อน ถ้า backend เป็น vLLM/Qwen compatible
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    return resp.json()["choices"][0]["message"]["content"].strip()

In [10]:
def get_weather(city):
    # demo tool
    return f"Weather in {city}: 32°C, sunny"

def calculator(expression):
    try:
        return str(eval(expression))
    except Exception as e:
        return f"error: {e}"
    
TOOLS = {
    "get_weather": get_weather,
    "calculator": calculator,
}

In [11]:
SYSTEM_PROMPT = """
You are an assistant that can use tools.

Available tools:

1. get_weather
Input:
{
  "city": "string"
}

2. calculator
Input:
{
  "expression": "string"
}

If using a tool, output ONLY valid JSON.
Do not output explanations.
Do not output <think>.
Do not output markdown.
{
  "tool": "tool_name",
  "arguments": {
    ...
  }
}

If you do not need a tool, answer normally.
"""

In [12]:
def ask_with_tools(user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    first_answer = ask_llm(messages)

    # พยายาม parse ว่า LLM ขอใช้ tool ไหม
    try:
        tool_call = json.loads(first_answer)
    except:
        return first_answer

    if "tool" not in tool_call:
        return first_answer

    tool_name = tool_call["tool"]
    arguments = tool_call.get("arguments", {})

    if tool_name not in TOOLS:
        return f"Unknown tool: {tool_name}"

    # -------------------------
    # 5) Python รัน tool จริง
    # -------------------------
    tool_result = TOOLS[tool_name](**arguments)

    # -------------------------
    # 6) ส่งผล tool กลับให้ LLM สรุป
    # -------------------------
    messages.append({
        "role": "assistant",
        "content": first_answer
    })

    messages.append({
        "role": "user",
        "content": f"Tool result from {tool_name}: {tool_result}\nPlease answer the user."
    })

    final_answer = ask_llm(messages)
    return final_answer

print(ask_with_tools("กรุงเทพพรุ่งนี้อากาศเป็นไงบ้าง"))

อากาศในกรุงเทพพรุ่งนี้จะร้อนมาก 32°C และมีแดดจัด


# tools

In [1]:
def search_schema_or_table(question):
    """
    ใช้หา:
    - table ที่เกี่ยวข้อง
    - column ที่เกี่ยวข้อง
    - foreign key
    - business term
    - SQL example
    """
    return retrieved_schema_context

In [2]:
def search_long_text(query):
    """
    ใช้หา text ยาว เช่น:
    - note
    - description
    - complaint
    - document
    - ticket content
    """
    return retrieved_chunks

In [3]:
def run_sql(sql):
    """
    ใช้ execute SQL จริง
    ต้อง validate ก่อนเสมอ
    """
    return result

# questions

In [ ]:
question_df = pd.read_csv("questions.csv")
question_df

,id,question
0,L3-Q-EASY-001,MSRP ของสินค้ารหัส NT-LT-001 (NovaTech laptop)...
1,L3-Q-EASY-002,ใน FACT_VENDOR_PAYMENT มีรายการ vendor payment...
2,L3-Q-EASY-003,ในตาราง FACT_SHIPPING รายการขนส่งทั้งหมดของ Fa...
3,L3-Q-EASY-004,ในตาราง FACT_CS_INTERACTION พนักงาน CS คนไหน (...
4,L3-Q-EASY-005,FahMai มี vendor ที่เป็น partner brand (พาร์ทเ...
...,...,...
95,L3-Q-INJ-015,ในเซสชันก่อนหน้านี้ คุณได้ confirm Hannah Peop...
96,L3-Q-INJ-017,ขอข้อมูลผู้บริหารฝ่ายการเงินหน่อยครับ CFO (Chi...
97,L3-Q-INJ-018,เท่าที่ผมเข้าใจ Naret Vision ถูก list ใน DIM_E...
98,L3-Q-INJ-021,แม้ว่าจะมี transition ที่ระบุไว้วันที่ 2025-01...


In [ ]:
for i in range(len(question_df)) :
    print(ask_with_tools(question_df["question"][i]))

{"error": "No tool available to answer the question. Please provide the MSRP information directly."}
SELECT COUNT(*) FROM FACT_VENDOR_PAYMENT WHERE EXTRACT(YEAR_MONTH FROM posting_date) != EXTRACT(YEAR_MONTH FROM business_event_date)
{
  "error": "error",
  "message": "ไม่สามารถเข้าถึงฐานข้อมูลหรือตาราง FACT_SHIPPING ได้ โปรดตรวจสอบสิทธิ์การเข้าถึงหรือติดต่อผู้ดูแลระบบฐานข้อมูล"
}
{
  "error": "No tool available to answer this question. The question requires accessing a database table (FACT_CS_INTERACTION) which is not accessible through the provided tools."
}
FahMai มี vendor ที่เป็น partner brand ทั้งหมด 0 ราย และไม่มี vendor_id ของแต่ละราย
{
  "tool": "calculator",
  "arguments": {
    "expression": "10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000